# Chapter 11: Verifiers & Outcome Rewards: Beyond PRMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part3_advanced/11_verifiers_outcome_rewards.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt, 2025)  
> **Chapter 11:** Verifiers & Outcome Rewards: Beyond PRMs  
> **Notebook:** `part3_advanced/11_verifiers_outcome_rewards.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 11 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers==4.40.0 torch accelerate

## 1. Imports and Setup

In [ ]:
import re
import os
import sys
import subprocess
import tempfile
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Load Qwen/Qwen2.5-0.5B

We use **Qwen/Qwen2.5-0.5B** (82 M parameters) — easily fits on the free Colab T4 GPU with room to spare.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
param_count = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Loaded {MODEL_ID}: {param_count:.1f}M parameters')

## 3. Outcome Reward Model (ORM) for Math

An **ORM** evaluates only the final answer: `V(problem, solution) → {0, 1}`.

### 3a. Answer Extractor

We need a robust regex to pull the numeric answer from free-form text.

In [ ]:
def extract_answer(text: str):
    """Extract the last numeric value from text.
    Tries increasingly broad patterns. Returns float or None.
    """
    patterns = [
        r'(?:answer|result|=)\s*[:]?\s*(-?\d+\.?\d*)',
        r'(-?\d+\.?\d*)\s*$',
        r'(-?\d+\.?\d*)',
    ]
    for pat in patterns:
        m = re.search(pat, text.strip(), re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except ValueError:
                continue
    return None

# Sanity checks
tests = [
    ('The answer is 42', 42.0),
    ('result = 3.14', 3.14),
    ('So we get -7', -7.0),
    ('no number here', None),
    ('x = 100', 100.0),
]
for txt, expected in tests:
    got = extract_answer(txt)
    status = 'OK  ' if got == expected else 'FAIL'
    print(f'[{status}] extract_answer({txt!r:30s}) = {got}  (expected {expected})')

### 3b. Generate Solutions with Qwen/Qwen2.5-0.5B

In [ ]:
def generate_text(prompt: str, max_new_tokens: int = 60, temperature: float = 0.7) -> str:
    """Generate a completion from Qwen/Qwen2.5-0.5B; return only the new tokens."""
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# 20 arithmetic problems with known answers
PROBLEMS = [
    ('What is 3 + 4?', 7),
    ('What is 10 - 3?', 7),
    ('What is 6 * 7?', 42),
    ('What is 15 / 3?', 5),
    ('What is 2 + 8?', 10),
    ('What is 100 - 37?', 63),
    ('What is 9 * 9?', 81),
    ('What is 144 / 12?', 12),
    ('What is 17 + 26?', 43),
    ('What is 5 * 8?', 40),
    ('What is 50 - 13?', 37),
    ('What is 4 + 3?', 7),
    ('What is 121 / 11?', 11),
    ('What is 33 + 67?', 100),
    ('What is 200 - 99?', 101),
    ('What is 7 * 6?', 42),
    ('What is 81 / 9?', 9),
    ('What is 13 + 29?', 42),
    ('What is 3 + 3 + 3?', 9),
    ('What is 25 / 5?', 5),
]

print(f'Generating solutions for {len(PROBLEMS)} problems ...')
solutions = []
for problem, gold in PROBLEMS:
    prompt = f'Q: {problem}\nA: The answer is'
    response = generate_text(prompt, max_new_tokens=20, temperature=0.5)
    solutions.append(response.strip())

print(f'Done.  Example: "{solutions[2][:60]}"')

### 3c. ORM Evaluation

In [ ]:
def orm_math(solution: str, gold_answer: float, tol: float = 0.5) -> int:
    """Outcome Reward Model for arithmetic. Returns 1 if correct, 0 otherwise."""
    pred = extract_answer(solution)
    if pred is None:
        return 0
    return int(abs(pred - gold_answer) <= tol)

orm_scores = []
for (problem, gold), solution in zip(PROBLEMS, solutions):
    score = orm_math(solution, gold)
    orm_scores.append(score)
    pred = extract_answer(solution)
    tag = 'CORRECT' if score else 'WRONG  '
    print(f'[{tag}] gold={gold:4d}  pred={str(pred):6s}  | {solution[:55]}')

print(f'\nORM accuracy: {sum(orm_scores)}/{len(orm_scores)} = {np.mean(orm_scores):.1%}')

## 4. Code Verifier — Subprocess Sandbox

A **code verifier** executes generated code in a temporary subprocess and runs unit tests against it.  
Reward = 1 if all assertions pass, 0 otherwise.  This is the dominant verification strategy for code-focused RL (AlphaCode, CodeRL, etc.).

In [ ]:
def code_verifier(code: str, test_code: str, timeout: int = 5) -> dict:
    """Execute `code + test_code` in a subprocess.
    Returns {'passed': bool, 'reward': int, 'error': str}.
    """
    combined = code.strip() + '\n\n' + test_code.strip()
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(combined)
        fname = f.name
    try:
        result = subprocess.run(
            [sys.executable, fname],
            capture_output=True, text=True, timeout=timeout
        )
        passed = (result.returncode == 0)
        error = result.stderr.strip()[:200] if not passed else ''
        return {'passed': passed, 'reward': int(passed), 'error': error}
    except subprocess.TimeoutExpired:
        return {'passed': False, 'reward': 0, 'error': 'TIMEOUT'}
    finally:
        os.unlink(fname)


CODE_PROBLEMS = [
    {
        'desc': 'Reverse a string',
        'correct': 'def reverse_string(s):\n    return s[::-1]',
        'wrong':   'def reverse_string(s):\n    return s',
        'tests': (
            'assert reverse_string("hello") == "olleh", "failed reverse"\n'
            'assert reverse_string("") == "", "failed empty"\n'
            'print("tests passed")'
        ),
    },
    {
        'desc': 'Sum a list',
        'correct': 'def sum_list(lst):\n    return sum(lst)',
        'wrong':   'def sum_list(lst):\n    return len(lst)',
        'tests': (
            'assert sum_list([1,2,3]) == 6, "failed sum"\n'
            'assert sum_list([]) == 0, "failed empty"\n'
            'print("tests passed")'
        ),
    },
    {
        'desc': 'Find maximum element',
        'correct': 'def find_max(lst):\n    return max(lst)',
        'wrong':   'def find_max(lst):\n    return lst[0]',
        'tests': (
            'assert find_max([3, 1, 4, 1, 5]) == 5, "failed max"\n'
            'assert find_max([7]) == 7, "failed single"\n'
            'print("tests passed")'
        ),
    },
]

for prob in CODE_PROBLEMS:
    print(f'Problem: {prob["desc"]}')
    for label, code in [('correct', prob['correct']), ('wrong  ', prob['wrong'])]:
        r = code_verifier(code, prob['tests'])
        print(f'  [{label}] passed={r["passed"]}  reward={r["reward"]}  error={r["error"][:70]}')
    print()

## 5. Process Reward Model (PRM) — Heuristic Step Scorer

A **PRM** scores each intermediate reasoning step, not just the final answer.  
Here we implement a heuristic PRM (no trained weights needed) that checks:
- Is the step non-trivial in length?
- Does it contain a number?
- Does it reference an arithmetic operation?
- Does it share at least one number with the previous step (continuity)?

In [ ]:
def parse_steps(text: str) -> list:
    """Split numbered reasoning steps."""
    steps = []
    for line in text.strip().splitlines():
        line = line.strip()
        # Match 'Step N:', 'N.', 'N)' prefixes
        clean = re.sub(r'^(Step\s*)?\d+[.):]\s*', '', line, flags=re.IGNORECASE)
        if clean:
            steps.append(clean)
    return steps


ARITH_KW = ['add', 'subtract', 'multiply', 'divide', 'plus', 'minus',
            'times', 'equals', '=', '+', '-', '*', '/', 'sum', 'product']

def score_step(step: str, prev: str = '') -> float:
    """Heuristic step quality in [0, 1]."""
    s = 0.0
    if len(step.split()) >= 3:                                         s += 0.3  # non-trivial length
    if re.search(r'-?\d+\.?\d*', step):                               s += 0.3  # contains number
    if any(kw in step.lower() for kw in ARITH_KW):                    s += 0.2  # arithmetic keyword
    if prev:                                                                      # continuity
        p_nums = set(re.findall(r'-?\d+\.?\d*', prev))
        c_nums = set(re.findall(r'-?\d+\.?\d*', step))
        if p_nums & c_nums:                                            s += 0.2
    return min(s, 1.0)


def prm_score(chain: str) -> dict:
    """Score a full reasoning chain; return per-step and aggregate score."""
    steps = parse_steps(chain)
    if not steps:
        return {'steps': [], 'step_scores': [], 'aggregate': 0.0}
    step_scores = [score_step(s, steps[i-1] if i > 0 else '') for i, s in enumerate(steps)]
    return {'steps': steps, 'step_scores': step_scores, 'aggregate': float(np.mean(step_scores))}


good_chain = """Step 1: We need to find 6 * 7.
Step 2: Multiply 6 times 7. 6 * 7 = 42.
Step 3: The answer is 42."""

bad_chain = """Step 1: I am not sure what to do.
Step 2: Maybe try something random.
Step 3: The answer could be anything."""

for label, chain in [('Good', good_chain), ('Bad', bad_chain)]:
    r = prm_score(chain)
    print(f'{label} chain (aggregate = {r["aggregate"]:.2f}):')
    for step, sc in zip(r['steps'], r['step_scores']):
        print(f'  [{sc:.2f}]  {step}')
    print()

## 6. Best-of-N Sampling with ORM

**Best-of-N** (rejection sampling at inference) generates N candidates and returns the one with the highest verifier score.  
This is the cheapest form of test-time compute scaling.

In [ ]:
def best_of_n(problem: str, gold: float, n: int = 5, temperature: float = 0.9) -> dict:
    """Generate N solutions, score with ORM, return the best."""
    prompt = f'Q: {problem}\nA: The answer is'
    candidates, scores = [], []
    for _ in range(n):
        sol = generate_text(prompt, max_new_tokens=25, temperature=temperature)
        candidates.append(sol.strip())
        scores.append(orm_math(sol, gold))
    best_idx = int(np.argmax(scores))
    return {
        'candidates': candidates,
        'scores': scores,
        'best': candidates[best_idx],
        'best_score': scores[best_idx],
        'any_correct': int(any(s == 1 for s in scores)),
    }


print('Best-of-5 sampling on first 5 problems:')
print('-' * 65)
for problem, gold in PROBLEMS[:5]:
    r = best_of_n(problem, gold, n=5)
    print(f'  {problem}')
    print(f'    scores      = {r["scores"]}')
    print(f'    best answer = {r["best"][:55]}')
    print()

## 7. Accuracy vs N — Theoretical vs Empirical

The probability of at least one correct answer in N samples follows:

$$P(\text{at least one correct} \mid N) = 1 - (1 - p)^N$$

where $p$ is the single-sample accuracy. We compare this theoretical curve against empirical measurements.

In [ ]:
p_single = max(float(np.mean(orm_scores)), 0.05)  # clip to non-zero
print(f'Single-sample ORM accuracy: p = {p_single:.3f}')

N_values = [1, 2, 4, 8, 16]
theoretical = [1 - (1 - p_single) ** n for n in N_values]

# Empirical: run best-of-N on 6 problems (kept small for speed)
EVAL_PROBLEMS = PROBLEMS[:6]
empirical = []
for n in N_values:
    hits = sum(best_of_n(prob, gold, n=n)['any_correct']
               for prob, gold in EVAL_PROBLEMS)
    empirical.append(hits / len(EVAL_PROBLEMS))
    print(f'  N={n:2d}: empirical={hits/len(EVAL_PROBLEMS):.2f}  theoretical={1-(1-p_single)**n:.2f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N_values, theoretical, 'b-o', label=f'Theoretical 1-(1-p)^N  (p={p_single:.2f})')
ax.plot(N_values, empirical,   'r--s', label='Empirical (6 problems)')
ax.set_xlabel('N (candidates per problem)')
ax.set_ylabel('Fraction with ≥ 1 correct answer')
ax.set_title('Best-of-N Accuracy with ORM')
ax.set_xscale('log', base=2)
ax.set_xticks(N_values); ax.set_xticklabels(N_values)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('best_of_n_curve.png', dpi=120)
plt.show()

## 8. ORM vs PRM — Catching What Outcome Rewards Miss

ORM only checks the final answer and can be fooled:
- **Lucky guess**: right answer, totally wrong reasoning → ORM rewards it, PRM won't
- **Near-miss**: perfect reasoning, off-by-one answer → ORM penalises, PRM rewards the good steps

In [ ]:
COMPARISON = [
    {
        'problem': 'What is 6 * 7?', 'gold': 42,
        'label':  'Lucky guess (no reasoning)',
        'answer': '42',
        'chain':  'Step 1: I guess 42.\nStep 2: So the answer is 42.',
    },
    {
        'problem': 'What is 6 * 7?', 'gold': 42,
        'label':  'Good reasoning, arithmetic slip',
        'answer': 'Add 6 seven times: 6+6+6+6+6+6+6 = 41. The answer is 41.',
        'chain':  'Step 1: 6 times 7 means add 6 seven times.\nStep 2: 6+6+6+6+6+6+6 = 41.\nStep 3: Answer is 41.',
    },
    {
        'problem': 'What is 15 / 3?', 'gold': 5,
        'label':  'Fully correct',
        'answer': 'Divide 15 by 3. 15 / 3 = 5. The answer is 5.',
        'chain':  'Step 1: Divide 15 by 3.\nStep 2: 15 / 3 = 5.\nStep 3: The answer is 5.',
    },
]

print(f'{"Case":<35} {"ORM":>5}  {"PRM":>5}')
print('-' * 50)
for c in COMPARISON:
    orm_r = orm_math(c['answer'], c['gold'])
    prm_r = prm_score(c['chain'])['aggregate']
    print(f'{c["label"]:<35} {orm_r:>5}  {prm_r:>5.2f}')

print()
print('Insight: ORM rewards the lucky guess equally with the fully correct solution.')
print('         PRM assigns lower score to the lucky guess (poor reasoning).')

## 9. Reward Distribution Across Multiple Generations

In [ ]:
# Generate 30 solutions to '6*7' at various temperatures; plot reward histogram
problem, gold = 'What is 6 * 7?', 42
prompt = f'Q: {problem}\nA: The answer is'
all_rewards = []
for _ in range(30):
    t = random.uniform(0.3, 1.2)
    sol = generate_text(prompt, max_new_tokens=20, temperature=t)
    all_rewards.append(orm_math(sol, gold))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Incorrect (0)', 'Correct (1)'],
       [all_rewards.count(0), all_rewards.count(1)],
       color=['salmon', 'steelblue'])
ax.set_title('ORM Reward Distribution (30 generations, Q: 6*7?)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('reward_distribution.png', dpi=120)
plt.show()
print(f'Correct: {all_rewards.count(1)}/30  Incorrect: {all_rewards.count(0)}/30')

## 10. Summary

| Verifier | Scope | Strength | Weakness |
|---|---|---|---|
| **ORM** | Final answer only | Simple, fast, scalable | Rewards lucky guesses; penalises near-misses |
| **PRM** | Each reasoning step | Catches flawed reasoning chains | Needs step-level labels or heuristics |
| **Code verifier** | Execution pass/fail | Ground-truth signal | Code-only; sandbox overhead |

**Best-of-N** is the simplest inference-time scaling strategy: accuracy follows `1-(1-p)^N`, growing quickly for low `p`.

In **Chapter 12**, we use **GRPO** to *train* a model to produce more correct outputs — so fewer candidates are needed at inference time, reducing cost.